# 🤖 Megaman RL Training Facility (The Dojo)

### 📋 Instructions
1. **Runtime:** Ensure you are using **T4 GPU** (Runtime -> Change runtime type).
2. **Run All:** Execute the cells below in order.
3. **Auth:** You will be asked to authorize Google Drive access for saving checkpoints.

In [ ]:
# 1. Initialize & Mount Drive
from google.colab import drive
import os

print('💾 Mounting Google Drive...')
drive.mount('/content/drive')

DRIVE_LOGS_DIR = '/content/drive/MyDrive/Megaman_Training_Logs'
os.makedirs(DRIVE_LOGS_DIR, exist_ok=True)
print(f'✅ Checkpoints will save to: {DRIVE_LOGS_DIR}')

REPO_URL = 'https://github.com/LoganStunts/Megaman-RL-Dojo.git'
REPO_NAME = 'Megaman-RL-Dojo'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull
    %cd ..
%cd {REPO_NAME}
print('✅ Repo Ready.')

In [ ]:
# 2. Install Infrastructure
!apt-get install -y libxcursor1 libxinerama1 libxrandr2 libxi6 libgl1-mesa-dev xvfb
!pip install godot-rl stable-baselines3 shimmy>=0.2.1 tensorboard huggingface_sb3 onnx
if not os.path.exists('Godot_v4.5.1-stable_linux.x86_64'):
    !wget https://github.com/godotengine/godot/releases/download/4.5.1-stable/Godot_v4.5.1-stable_linux.x86_64.zip
    !unzip -o Godot_v4.5.1-stable_linux.x86_64.zip
    !chmod +x Godot_v4.5.1-stable_linux.x86_64
print('✅ Infrastructure Ready.')

In [ ]:
# 3. Switch to Payload
TARGET_BRANCH = 'payload/main'
!git fetch origin
!git checkout {TARGET_BRANCH}
!git pull origin {TARGET_BRANCH}
print(f'✅ Switched to {TARGET_BRANCH}.')

In [ ]:
# 4. Ignite Training (Optimized Custom Script)
import subprocess, time
subprocess.run(['pkill', '-f', 'Godot'])
time.sleep(2)

GODOT_BIN = '/content/Megaman-RL-Dojo/Godot_v4.5.1-stable_linux.x86_64'

print('📦 Importing assets...')
with open('godot_import.log', 'w') as f:
    # We use the full path to the project folder inside the container
    subprocess.run(['xvfb-run', '-a', GODOT_BIN, '--headless', '--editor', '--quit', '--path', '/content/Megaman-RL-Dojo'], stdout=f, stderr=f)

print('🧠 Starting Optimized Trainer...')
print(f'   -> Logging to: {DRIVE_LOGS_DIR}')

# EXECUTE THE CUSTOM SCRIPT
!xvfb-run -a python3 train_optimized.py --env_path={GODOT_BIN} --n_parallel=8 --speedup=8 --save_path={DRIVE_LOGS_DIR} --timesteps=1000000